# Woche 05: Statistik in der Genomik mit Python

**Block A · Woche 05 · Selbststudium · Starter (Skeleton)**

Dieses Notebook enthält die Python-Beispiele aus dem Skriptum *„Statistik in der Genomik: Warum gute Daten allein
nicht reichen“* (Woche 05). In Woche 5 gibt es keine Pflichtübung; das Notebook dient zum Nachvollziehen und Üben.

## Lernziele
- Eine breite Tabelle in **Tidy Data** umformen (Skriptum 4.2)
- Kennzahlen von **Anscombes Quartett** nachrechnen und plotten (Skriptum 5.1)
- Die **Welch-t-Statistik** und einen **Permutationstest** selbst programmieren (Skriptum 6.1, 6.2, 6.4)
- **Bonferroni** und **Benjamini-Hochberg** implementieren und an einer Simulation mit 10.000 Genen testen (Skriptum 7)

## Zu implementierende Funktionen
| Teil | Funktion | Skriptum |
|---|---|---|
| 1 | `wide_to_tidy` | 4.2 |
| 2 | `summary_stats` | 5.1 |
| 3 | `welch_t`, `perm_p`, `exact_perm_p` | 6.1 bis 6.4 |
| 4 | `bonferroni`, `benjamini_hochberg` | 7.2 bis 7.4 |

> Im **Starter** werfen diese Funktionen `NotImplementedError`. Ersetze den Rumpf durch deine eigene Implementierung
> und führe danach die Test-Zelle des jeweiligen Teils aus, bis alle Tests grün sind. Die **Musterlösung** zeigt eine
> mögliche Lösung. Die Beispieldaten in Teil 1 und 3 sind fiktiv.

## 0. Setup
Führe diese Zelle zuerst aus. Fehlende Pakete werden einmalig installiert.

In [ ]:
# Benötigte Pakete (werden bei Bedarf einmalig installiert)
import importlib, subprocess, sys
for package in ["numpy", "scipy", "pandas", "statsmodels", "matplotlib", "ipytest", "pytest"]:
    try:
        importlib.import_module(package)
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package])

import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pytest
from scipy import stats
from statsmodels.stats.multitest import multipletests

import ipytest
ipytest.autoconfig()

## 1. Tidy Data (Skriptum 4.2)

Expressionsdaten liegen oft „breit“ vor: eine Zeile pro Gen, eine Spalte pro Probe. Tidy Data verlangt dagegen eine
Variable pro Spalte und eine Beobachtung pro Zeile (Leek, *How to share data with a statistician*; Wickham 2014).

**Aufgabe:** Schreibe `wide_to_tidy(wide)`. Die Eingabe hat eine Spalte `gene` und beliebig viele Probenspalten.
Das Ergebnis hat genau die Spalten `gene`, `sample`, `log2_expr` und eine Zeile pro Messung.

*Tipp:* `DataFrame.melt` (pandas-Dokumentation).

In [ ]:
def wide_to_tidy(wide: pd.DataFrame) -> pd.DataFrame:
    """Breite Tabelle (gene + eine Spalte pro Probe) -> Tidy Data (gene, sample, log2_expr)."""
    # Tipp: wide.melt(id_vars=..., var_name=..., value_name=...)
    raise NotImplementedError("TODO: implement wide_to_tidy")

In [ ]:
wide = pd.DataFrame({"gene": ["BRCA1", "TP53"],
                     "sample_1": [7.9, 5.1],
                     "sample_2": [8.4, 5.3]})
print(wide, end="\n\n")
print(wide_to_tidy(wide))

In [ ]:
%%ipytest

def test_tidy_columns_and_rows():
    wide = pd.DataFrame({"gene": ["BRCA1", "TP53"], "sample_1": [7.9, 5.1], "sample_2": [8.4, 5.3]})
    long = wide_to_tidy(wide)
    assert list(long.columns) == ["gene", "sample", "log2_expr"]
    assert len(long) == 4


def test_tidy_values_preserved():
    wide = pd.DataFrame({"gene": ["BRCA1", "TP53"], "sample_1": [7.9, 5.1], "sample_2": [8.4, 5.3]})
    long = wide_to_tidy(wide).set_index(["gene", "sample"])
    assert long.loc[("TP53", "sample_2"), "log2_expr"] == 5.3
    assert long.loc[("BRCA1", "sample_1"), "log2_expr"] == 7.9


def test_tidy_any_number_of_samples():
    wide = pd.DataFrame({"gene": ["MYC"], "p1": [1.0], "p2": [2.0], "p3": [3.0]})
    long = wide_to_tidy(wide)
    assert len(long) == 3
    assert sorted(long["sample"]) == ["p1", "p2", "p3"]

## 2. Anscombes Quartett (Skriptum 5.1)

Die vier Datensätze stammen aus Anscombe (1973). Laut Skriptum haben alle vier fast identische Kennzahlen.

**Aufgabe:** Schreibe `summary_stats(x, y)`, die ein Dictionary mit folgenden Schlüsseln liefert:
`mean_x`, `var_x`, `mean_y`, `var_y` (Stichprobenvarianz, also `ddof=1`), `r` (Korrelationskoeffizient),
`intercept` und `slope` der Regressionsgeraden y = intercept + slope · x.

*Tipps:* `np.mean`, `np.var(..., ddof=1)`, `np.corrcoef`, `np.polyfit(x, y, 1)` (liefert zuerst die Steigung).

In [ ]:
X = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
ANSCOMBE = {
    "I":   (X, [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    "II":  (X, [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    "III": (X, [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    "IV":  ([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
            [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
}

In [ ]:
def summary_stats(x, y) -> dict:
    """Mittelwerte, Stichprobenvarianzen, Korrelation und Regressionsgerade von (x, y)."""
    # Tipp: Dictionary mit mean_x, var_x, mean_y, var_y, r, intercept, slope zurückgeben
    raise NotImplementedError("TODO: implement summary_stats")

In [ ]:
# Kennzahlen als Tabelle und die vier Plots: erst der Plot zeigt die Unterschiede
print(pd.DataFrame({name: summary_stats(x, y) for name, (x, y) in ANSCOMBE.items()}).round(3))

fig, axs = plt.subplots(1, 4, figsize=(14, 3.2), sharey=True)
for ax, (name, (x, y)) in zip(axs, ANSCOMBE.items()):
    k = summary_stats(x, y)
    gx = np.array([3, 20])
    ax.plot(gx, k["intercept"] + k["slope"] * gx, color="grey")
    ax.scatter(x, y)
    ax.set_title(f"Dataset {name}")
    ax.set_xlabel("x")
axs[0].set_ylabel("y")
plt.tight_layout()
plt.show()

In [ ]:
%%ipytest

def test_summary_stats_simple():
    k = summary_stats([1, 2, 3], [2, 4, 6])
    assert k["mean_x"] == pytest.approx(2.0)
    assert k["var_x"] == pytest.approx(1.0)        # Stichprobenvarianz (ddof=1)
    assert k["r"] == pytest.approx(1.0)
    assert k["slope"] == pytest.approx(2.0)
    assert k["intercept"] == pytest.approx(0.0, abs=1e-9)


@pytest.mark.parametrize("name", ["I", "II", "III", "IV"])
def test_anscombe_same_stats(name):
    x, y = ANSCOMBE[name]
    k = summary_stats(x, y)
    assert k["mean_x"] == pytest.approx(9.0)
    assert k["var_x"] == pytest.approx(11.0)
    assert k["mean_y"] == pytest.approx(7.50, abs=0.01)
    assert k["var_y"] == pytest.approx(4.12, abs=0.01)
    assert k["r"] == pytest.approx(0.816, abs=0.002)
    assert k["intercept"] == pytest.approx(3.0, abs=0.01)
    assert k["slope"] == pytest.approx(0.5, abs=0.001)

## 3. t-Statistik und Permutationstest (Skriptum 6.1 bis 6.4)

Leek verwendet die t-Statistik

$$t = \frac{\bar{Y} - \bar{X}}{\sqrt{\frac{s_Y^2}{N} + \frac{s_X^2}{M}}}$$

mit den Gruppenmittelwerten, den Stichprobenvarianzen ($s^2$, `ddof=1`) und den Gruppengrößen $N$ und $M$.
Das ist die **Welch-t-Statistik**, die `scipy.stats.ttest_ind(a, b, equal_var=False)` berechnet.

Der **Permutations-p-Wert** ist der Anteil der permutierten Statistiken, die betragsmäßig mindestens so groß sind
wie die beobachtete: $p = \#\{|S_{perm}| \ge |S_{obs}|\} \,/\, \text{Anzahl Permutationen}$.

**Aufgaben:**
1. `welch_t(a, b)`: berechne t = (Mittelwert a − Mittelwert b) / √(s²_a/n_a + s²_b/n_b) **ohne** SciPy.
2. `perm_p(x, y, n_perm, seed)`: Gruppenlabels `n_perm`-mal zufällig vertauschen (Monte-Carlo-Permutation).
3. `exact_perm_p(x, y)`: **alle** möglichen Aufteilungen durchgehen (`itertools.combinations`).

In [ ]:
def welch_t(a, b) -> float:
    """Welch-t-Statistik: (mean(a) - mean(b)) / sqrt(var(a)/n_a + var(b)/n_b), Varianzen mit ddof=1."""
    # Tipp: Formel aus Skriptum 6.1, Varianzen mit ddof=1
    raise NotImplementedError("TODO: implement welch_t")


def perm_p(x, y, n_perm=10_000, seed=1) -> float:
    """Monte-Carlo-Permutationstest (zweiseitig) mit der Welch-t-Statistik."""
    # Tipp: Werte zusammenlegen, n_perm-mal mischen (rng.shuffle), Anteil |t_perm| >= |t_obs| berechnen
    raise NotImplementedError("TODO: implement perm_p")


def exact_perm_p(x, y) -> float:
    """Exakter Permutationstest: alle Aufteilungen der Werte in Gruppen der Größe len(x) und len(y)."""
    # Tipp: itertools.combinations(range(n), len(x)) liefert alle Aufteilungen
    raise NotImplementedError("TODO: implement exact_perm_p")

In [ ]:
# fiktive log2-Expressionswerte eines Gens (je 5 Proben), wie im Skriptum 6.4
diseased = np.array([7.9, 8.4, 8.1, 8.8, 8.3])
healthy  = np.array([7.2, 7.6, 7.0, 7.9, 7.4])

t, p = stats.ttest_ind(diseased, healthy, equal_var=False)
print(f"SciPy Welch t-test: t = {t:.2f}, p = {p:.4f}")              # Skriptum: t = 4.04, p = 0.0037
print(f"own welch_t:        t = {welch_t(diseased, healthy):.2f}")
print(f"permutation p (Monte Carlo, 10,000x): {perm_p(diseased, healthy):.4f}")   # Skriptum: 0.0148
print(f"exact permutation p:                  {exact_perm_p(diseased, healthy):.4f}")  # 4/252

# Gegenprobe mit SciPy: alle 252 Aufteilungen exakt
res = stats.permutation_test((diseased, healthy),
                             lambda a, b: stats.ttest_ind(a, b, equal_var=False).statistic,
                             permutation_type="independent", n_resamples=np.inf)
print(f"scipy.stats.permutation_test:         {res.pvalue:.4f}")

In [ ]:
%%ipytest

DISEASED = np.array([7.9, 8.4, 8.1, 8.8, 8.3])
HEALTHY  = np.array([7.2, 7.6, 7.0, 7.9, 7.4])


def test_welch_t_matches_scipy():
    expected = stats.ttest_ind(DISEASED, HEALTHY, equal_var=False).statistic
    assert welch_t(DISEASED, HEALTHY) == pytest.approx(expected)
    assert welch_t(DISEASED, HEALTHY) == pytest.approx(4.04, abs=0.01)


def test_welch_t_sign():
    assert welch_t(HEALTHY, DISEASED) == pytest.approx(-welch_t(DISEASED, HEALTHY))


def test_welch_t_unequal_sizes():
    a, b = [1.0, 2.0, 4.0], [2.0, 3.0, 5.0, 9.0]
    assert welch_t(a, b) == pytest.approx(stats.ttest_ind(a, b, equal_var=False).statistic)


def test_perm_p_close_to_exact():
    p = perm_p(DISEASED, HEALTHY, n_perm=10_000, seed=1)
    assert 0.0 <= p <= 1.0
    assert p == pytest.approx(4 / 252, abs=0.006)


def test_perm_p_no_difference():
    identical = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
    assert perm_p(identical, identical.copy(), n_perm=200) == 1.0


def test_exact_perm_p():
    assert exact_perm_p(DISEASED, HEALTHY) == pytest.approx(4 / 252)


def test_exact_perm_p_smallest_value():
    # vollständig getrennte Gruppen: nur die beobachtete und die gespiegelte Aufteilung sind so extrem
    assert exact_perm_p([10, 11, 12, 13, 14], [1, 2, 3, 4, 5]) == pytest.approx(2 / 252)

## 4. Multiples Testen (Skriptum 7)

- **Bonferroni** kontrolliert die FWER: Bei m Tests gilt ein Test als Treffer, wenn p ≤ α/m.
- **Benjamini-Hochberg** kontrolliert die FDR: p-Werte aufsteigend sortieren, das größte k mit
  p₍ₖ₎ ≤ k·α/m suchen und die k kleinsten p-Werte zu Treffern erklären (Benjamini & Hochberg 1995).

**Aufgabe:** Implementiere beide Verfahren. Beide Funktionen liefern ein boolesches NumPy-Array
(True = Treffer) in der **ursprünglichen Reihenfolge** der p-Werte.

*Tipps:* `np.argsort`, `np.arange(1, m + 1)`, `np.nonzero`.

In [ ]:
def bonferroni(p, alpha=0.05) -> np.ndarray:
    """Bonferroni-Korrektur: Treffer, wenn p <= alpha / m."""
    # Tipp: boolesches Array mit p <= alpha / m
    raise NotImplementedError("TODO: implement bonferroni")


def benjamini_hochberg(p, alpha=0.05) -> np.ndarray:
    """Benjamini-Hochberg-Verfahren (FDR-Kontrolle)."""
    # Tipp: sortieren, Grenzen k*alpha/m berechnen, größtes k finden, die k kleinsten markieren
    raise NotImplementedError("TODO: implement benjamini_hochberg")

In [ ]:
# Beispiel aus Kontrollfrage 9 des Skriptums
p_example = [0.001, 0.008, 0.012, 0.041, 0.20]
print("Bonferroni:        ", bonferroni(p_example))
print("Benjamini-Hochberg:", benjamini_hochberg(p_example))

### Simulation: 10.000 Gene, nur 500 davon wirklich verändert (Skriptum 7.1 und 7.4)

Bei den ersten 500 Genen ist die zweite Gruppe um zwei Standardabweichungen verschoben, bei allen anderen gilt die
Nullhypothese. Weil wir die Wahrheit kennen, können wir die Fehler zählen. Erwartete Ausgabe laut Skriptum:
ohne Korrektur 966 Treffer (469 falsch), Bonferroni 55 (0 falsch), Benjamini-Hochberg 417 (26 falsch).

In [ ]:
rng = np.random.default_rng(2026)
a = rng.normal(0, 1, size=(10_000, 10))
b = rng.normal(0, 1, size=(10_000, 10))
b[:500] += 2                       # nur 500 Gene verändert

p = stats.ttest_ind(a, b, axis=1).pvalue
true_effect = np.zeros(10_000, dtype=bool)
true_effect[:500] = True

results = {
    "uncorrected": p < 0.05,
    "Bonferroni (own)": bonferroni(p),
    "Bonferroni (statsmodels)": multipletests(p, alpha=0.05, method="bonferroni")[0],
    "BH (own)": benjamini_hochberg(p),
    "BH (statsmodels)": multipletests(p, alpha=0.05, method="fdr_bh")[0],
}
table = pd.DataFrame({name: {"hits": sig.sum(),
                             "false positives": (sig & ~true_effect).sum(),
                             "true effects found": (sig & true_effect).sum()}
                      for name, sig in results.items()}).T
print(table)

fig, axs = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
bins = np.linspace(0, 1, 21)
axs[0].hist(p[~true_effect], bins=bins)
axs[0].set_title("9,500 genes without effect")
axs[1].hist(p, bins=bins)
axs[1].set_title("all 10,000 genes (500 changed)")
for ax in axs:
    ax.axvline(0.05, color="red")
    ax.set_xlabel("p-value")
axs[0].set_ylabel("number of genes")
plt.tight_layout()
plt.show()

In [ ]:
%%ipytest

P_EXAMPLE = [0.001, 0.008, 0.012, 0.041, 0.20]


def test_bonferroni_example():
    assert list(bonferroni(P_EXAMPLE)) == [True, True, False, False, False]


def test_bh_example():
    assert list(benjamini_hochberg(P_EXAMPLE)) == [True, True, True, False, False]


def test_bh_keeps_order():
    # unsortierte Eingabe: Ergebnis muss zur ursprünglichen Reihenfolge passen
    assert list(benjamini_hochberg([0.20, 0.012, 0.001, 0.041, 0.008])) == [False, True, True, False, True]


def test_no_hits():
    assert not benjamini_hochberg([0.5, 0.7, 0.9]).any()
    assert not bonferroni([0.5, 0.7, 0.9]).any()


@pytest.mark.parametrize("seed", [0, 1, 2])
def test_matches_statsmodels(seed):
    p = np.random.default_rng(seed).uniform(size=2_000) ** 3
    assert (bonferroni(p) == multipletests(p, alpha=0.05, method="bonferroni")[0]).all()
    assert (benjamini_hochberg(p) == multipletests(p, alpha=0.05, method="fdr_bh")[0]).all()

## Quellen
- Skriptum Woche 05: *Statistik in der Genomik* (Quellenverzeichnis dort, Nummern [1] bis [27])
- Leek, J.: Foliensätze zu Woche 05 (Coursera „Introduction to Genomic Technologies“, CC BY-NC-SA)
- Leek, J. T.: How to share data with a statistician. github.com/jtleek/datasharing
- Wickham, H.: Tidy Data. J. Stat. Softw. 59(10) (2014)
- Anscombe, F. J.: Graphs in Statistical Analysis. Am. Stat. 27(1), 17–21 (1973)
- Benjamini, Y.; Hochberg, Y.: Controlling the false discovery rate. J. R. Stat. Soc. B 57(1), 289–300 (1995)
- Dokumentation: pandas `DataFrame.melt`, SciPy `ttest_ind` und `permutation_test`, statsmodels `multipletests`